In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/raw/vehicles.csv")

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (50242, 84)


In [ ]:
fuel_type = (
    df["fuelType"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

fuel_type_1 = (
    df["fuelType1"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

fuel_type_2 = (
    df["fuelType2"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

In [ ]:
fuel_type

0        regular gasoline
1        regular gasoline
2        regular gasoline
3        regular gasoline
4        premium gasoline
               ...       
50237    regular gasoline
50238    regular gasoline
50239    regular gasoline
50240    regular gasoline
50241    premium gasoline
Name: fuelType1, Length: 50242, dtype: str

In [ ]:
fuel_type_2

In [ ]:
# Keep only vehicles entirely powered by electricity:

bev_mask = ((fuel_type == "electricity") & (fuel_type_1 == "electricity")
    & (fuel_type_2 == "")
)

bev_df = df[bev_mask].copy()

print(f"All vehicle records: {len(df)}")
print(f"Battery electric vehicle records: {len(bev_df)}")

All vehicle records: 50,242
Battery electric vehicle records: 1,572


In [5]:
bev_mask

0        False
1        False
2        False
3        False
4        False
         ...  
50237    False
50238    False
50239    False
50240    False
50241    False
Length: 50242, dtype: bool

In [ ]:
# Keep only vehicles entirely powered by electricity:

bev_df[
    ["fuelType", "fuelType1", "fuelType2"]
].value_counts(dropna=False)

fuelType     fuelType1    fuelType2
Electricity  Electricity  NaN          1572
Name: count, dtype: int64

In [11]:
bev_columns = [
    "id",
    "year",
    "make",
    "model",
    "VClass",
    "drive",
    "trany",
    "fuelType",
    "range",
    "city08",
    "highway08",
    "comb08",
    "charge120",
    "charge240",
    "evMotor",
    "co2TailpipeGpm",
    "fuelCost08",
    "youSaveSpend",
]

bev_df[bev_columns].head(15)

,id,year,make,model,VClass,drive,trany,fuelType,range,city08,highway08,comb08,charge120,charge240,evMotor,co2TailpipeGpm,fuelCost08,youSaveSpend
7138,16423,2000,Nissan,Altra EV,Midsize Station Wagons,NaN,NaN,Electricity,90,81,91,85,0.0,0.0,62 KW AC Induction,0.0,900,6500
7139,16424,2000,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Electricity,88,81,64,72,0.0,0.0,50 KW DC,0.0,1050,5750
8143,17328,2001,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Electricity,88,81,64,72,0.0,0.0,50 KW DC,0.0,1050,5750
8144,17329,2001,Ford,Th!nk,Two Seaters,NaN,NaN,Electricity,29,74,58,65,0.0,0.0,27 KW AC Induction,0.0,1150,5250
8146,17330,2001,Ford,Explorer USPS Electric,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Electricity,38,45,33,39,0.0,0.0,67 KW AC Induction,0.0,1950,1250
8147,17331,2001,Nissan,Hyper-Mini,Two Seaters,NaN,NaN,Electricity,33,84,66,75,0.0,0.0,24 KW AC Synchronous,0.0,1000,6000
9212,18290,2002,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Electricity,95,87,69,78,0.0,0.0,50 KW DC,0.0,950,6250
9213,18291,2002,Ford,Explorer USPS Electric,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Electricity,38,45,33,39,0.0,0.0,67 KW AC Induction,0.0,1950,1250
10329,19296,2003,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Electricity,95,87,69,78,0.0,0.0,50 KW DC,0.0,950,6250
23016,30965,2001,Ford,Ranger Pickup 2WD,Standard Pickup Trucks 2WD,2-Wheel Drive,Automatic (A1),Electricity,50,62,54,58,0.0,0.0,67 KW AC Induction,0.0,1300,4500


In [ ]:
# This gives you an initial view of the kind of questions an assistant will answer.

bev_df[
    [
        "year",
        "make",
        "model",
        "range",
        "comb08",
        "charge240",
        "VClass",
    ]
].sort_values("range", ascending=False).head(20)

,year,make,model,range,comb08,charge240,VClass
37898,2022,Lucid,Air Dream R AWD w/19 inch wheels,520,125,13.0,Large Cars
39902,2023,Lucid,Air G Touring XR AWD with 19 inch wheels,516,131,13.0,Large Cars
37900,2022,Lucid,Air G Touring AWD w/19 inch wheels,516,131,13.0,Large Cars
41597,2024,Lucid,Air G Touring XR AWD with19 inch wheels,516,129,13.0,Large Cars
43944,2026,Lucid,Air G Touring XR AWD with19 inch wheels,512,128,13.0,Large Cars
42189,2025,Lucid,Air G Touring XR AWD with 19 inch wheels,512,128,13.0,Large Cars
43581,2026,Chevrolet,Silverado EV Max Range WT (19kW Charger),493,68,13.8,Standard Pickup Trucks 4WD
42549,2025,Chevrolet,Silverado EV 8WT,492,68,13.3,Standard Pickup Trucks 4WD
41598,2024,Lucid,Air G Touring XR AWD with 20 inch wheels,485,121,13.0,Large Cars
37899,2022,Lucid,Air Dream R AWD w/21 inch wheels,481,116,13.0,Large Cars


In [16]:
processed_path = Path("../data/processed/bev_vehicles.csv")

bev_clean = pd.read_csv(processed_path)

print(f"Processed BEV dataset shape: {bev_clean.shape}")

bev_clean.head()

Processed BEV dataset shape: (1572, 20)


,id,year,make,model,vehicle_class,drive,transmission,vehicle_type,fuel_type,electric_range_miles,city_mpge,highway_mpge,combined_mpge,charge_120v_hours,charge_240v_hours,ev_motor,annual_fuel_cost_usd,tailpipe_co2_gpm,vehicle_name,document_text
0,16423,2000,Nissan,Altra EV,Midsize Station Wagons,NaN,NaN,Battery Electric Vehicle (BEV),Electricity,90,81,91,85,NaN,NaN,62 KW AC Induction,900.0,0.0,2000 Nissan Altra EV,Vehicle: 2000 Nissan Altra EV\nFuelEconomy.gov...
1,16424,2000,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Battery Electric Vehicle (BEV),Electricity,88,81,64,72,NaN,NaN,50 KW DC,1050.0,0.0,2000 Toyota RAV4 EV,Vehicle: 2000 Toyota RAV4 EV\nFuelEconomy.gov ...
2,17328,2001,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Battery Electric Vehicle (BEV),Electricity,88,81,64,72,NaN,NaN,50 KW DC,1050.0,0.0,2001 Toyota RAV4 EV,Vehicle: 2001 Toyota RAV4 EV\nFuelEconomy.gov ...
3,17329,2001,Ford,Th!nk,Two Seaters,NaN,NaN,Battery Electric Vehicle (BEV),Electricity,29,74,58,65,NaN,NaN,27 KW AC Induction,1150.0,0.0,2001 Ford Th!nk,Vehicle: 2001 Ford Th!nk\nFuelEconomy.gov vehi...
4,17330,2001,Ford,Explorer USPS Electric,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,Battery Electric Vehicle (BEV),Electricity,38,45,33,39,NaN,NaN,67 KW AC Induction,1950.0,0.0,2001 Ford Explorer USPS Electric,Vehicle: 2001 Ford Explorer USPS Electric\nFue...


In [17]:
bev_clean["vehicle_type"].value_counts()

vehicle_type
Battery Electric Vehicle (BEV)    1572
Name: count, dtype: int64

In [ ]:
# Check high-range vehicles:

bev_clean[
    [
        "vehicle_name",
        "electric_range_miles",
        "combined_mpge",
        "charge_240v_hours",
        "vehicle_class",
    ]
].sort_values(
    "electric_range_miles",
    ascending=False,
).head(20)

,vehicle_name,electric_range_miles,combined_mpge,charge_240v_hours,vehicle_class
297,2022 Lucid Air Dream R AWD w/19 inch wheels,520,125,13.0,Large Cars
462,2023 Lucid Air G Touring XR AWD with 19 inch w...,516,131,13.0,Large Cars
299,2022 Lucid Air G Touring AWD w/19 inch wheels,516,131,13.0,Large Cars
677,2024 Lucid Air G Touring XR AWD with19 inch wh...,516,129,13.0,Large Cars
1281,2026 Lucid Air G Touring XR AWD with19 inch wh...,512,128,13.0,Large Cars
841,2025 Lucid Air G Touring XR AWD with 19 inch w...,512,128,13.0,Large Cars
1143,2026 Chevrolet Silverado EV Max Range WT (19kW...,493,68,13.8,Standard Pickup Trucks 4WD
947,2025 Chevrolet Silverado EV 8WT,492,68,13.3,Standard Pickup Trucks 4WD
678,2024 Lucid Air G Touring XR AWD with 20 inch w...,485,121,13.0,Large Cars
298,2022 Lucid Air Dream R AWD w/21 inch wheels,481,116,13.0,Large Cars


In [19]:
print(bev_clean.loc[0, "document_text"])

Vehicle: 2000 Nissan Altra EV
FuelEconomy.gov vehicle ID: 16423
Vehicle type: Battery Electric Vehicle (BEV)
Vehicle class: Midsize Station Wagons
Electric range: 90 miles
City efficiency: 81 MPGe
Highway efficiency: 91 MPGe
Combined efficiency: 85 MPGe
Electric motor: 62 KW AC Induction
Annual energy cost estimate: 900 USD
Tailpipe CO2 emissions: 0 grams per mile
Source: U.S. Department of Energy FuelEconomy.gov dataset.


In [ ]:
# Add BEV-only quality checks

assert bev_clean["id"].is_unique
assert bev_clean["vehicle_name"].notna().all()
assert bev_clean["document_text"].notna().all()

assert (
    bev_clean["vehicle_type"]
    == "Battery Electric Vehicle (BEV)"
).all()

assert (
    bev_clean["fuel_type"]
    == "Electricity"
).all()

print("All BEV-only quality checks passed.")

All BEV-only quality checks passed.
